In [1]:
# =========================================
# 05_prepare_feature_matrix_pooled.ipynb
# Build final pooled X / y / groups matrix for grouped modeling
# =========================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 200)

WORK_DIR = r"C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026"

MASTER_FILE = os.path.join(WORK_DIR, "interim", "master_table.parquet")

INTERIM_DIR = os.path.join(WORK_DIR, "interim")
META_DIR = os.path.join(WORK_DIR, "metadata")
REPORT_DIR = os.path.join(WORK_DIR, "reports")
ARTIFACT_DIR = os.path.join(WORK_DIR, "artifacts", "05_feature_matrix")

for p in [INTERIM_DIR, META_DIR, REPORT_DIR, ARTIFACT_DIR]:
    os.makedirs(p, exist_ok=True)

print("MASTER_FILE:", MASTER_FILE)

MASTER_FILE: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\master_table.parquet


In [2]:
master = pd.read_parquet(MASTER_FILE)

print("Master shape:", master.shape)
master.head(2)

Master shape: (1835, 69)


,raw_row_id,Ref_DOI_number,Ref_publication_date,publication_year,Cell_architecture,Stability_protocol,Encapsulation,Stability_PCE_T80,T80_raw,T80_clean,T80_log1p,architecture_family,is_nip,is_pin,is_other_arch,etl_family,htl_family,backcontact_family,etl_has_tio2,etl_has_sno2,etl_has_pcbm,etl_has_c60,etl_has_zno,htl_has_spiro,htl_has_ptaa,htl_has_pedot,htl_has_niox,htl_has_p3ht,back_has_au,back_has_ag,back_has_al,back_has_carbon,has_perovskite_additives,has_etl_additives,has_htl_additives,band_gap_ev,perovskite_thickness_nm,etl_thickness_nm,cell_area_measured_cm2,n_cells_per_substrate,band_gap_ev_missing,perovskite_thickness_nm_missing,etl_thickness_nm_missing,cell_area_measured_cm2_missing,encapsulation_flag,encapsulation_missing,protocol_family,protocol_has_l,protocol_has_d,protocol_has_o,bias_family,bias_is_mpp,bias_is_oc,bias_is_sc,bias_mentions_dark,light_intensity_suns,light_intensity_missing,light_bin,is_dark_condition,is_approx_1sun,is_high_light,temperature_c,temperature_missing,rh_pct,rh_missing,is_room_temperature,is_hot_test,is_dry_condition,is_humid_condition
0,27,10.1039/c9ta01893j,20/03/2019,2019,nip,ISOS-L-1,False,200.0,200.0,200.0,5.303305,nip,1,0,0,tio2,spiro_ometad,au,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,1.59,NaN,NaN,0.06,0.0,0,1,1,0,0.0,0,isos_l,1,0,0,mpp,1,0,0,0,100.0,0,high_light,0,0,1,25.0,0,NaN,1,1,0,0,0
1,45,10.1002/aenm.201803587,06/03/2019,2019,nip,ISOS-L-1,False,150.0,150.0,150.0,5.017280,nip,1,0,0,sno2,spiro_ometad,au,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,NaN,500.0,25.0,0.16,0.0,1,0,0,0,0.0,0,isos_l,1,0,0,mpp,1,0,0,0,100.0,0,high_light,0,0,1,25.0,0,NaN,1,1,0,0,0


In [3]:
TARGET_COL = "T80_log1p"
RAW_TARGET_COL = "T80_clean"
GROUP_COL = "Ref_DOI_number"
ROW_ID_COL = "raw_row_id"

required_cols = [TARGET_COL, RAW_TARGET_COL, GROUP_COL, ROW_ID_COL]
missing_required = [c for c in required_cols if c not in master.columns]

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

print("Target, raw target, group, and row ID columns are present.")

Target, raw target, group, and row ID columns are present.


In [4]:
forbidden_feature_cols = [
    # target / outcome columns
    "Stability_PCE_T80",
    "T80_raw",
    "T80_clean",
    "T80_log1p",

    # identifiers / grouping / metadata
    "raw_row_id",
    "Ref_DOI_number",
    "Ref_publication_date",
    "publication_year",

    # raw label columns
    "Cell_architecture",
    "Stability_protocol",
    "Encapsulation",
]

# remove missingness proxy columns from predictors
missing_proxy_cols = [c for c in master.columns if c.endswith("_missing")]

forbidden_feature_cols = [
    c for c in (forbidden_feature_cols + missing_proxy_cols)
    if c in master.columns
]

print("Forbidden feature columns:")
for c in forbidden_feature_cols:
    print(" -", c)

Forbidden feature columns:
 - Stability_PCE_T80
 - T80_raw
 - T80_clean
 - T80_log1p
 - raw_row_id
 - Ref_DOI_number
 - Ref_publication_date
 - publication_year
 - Cell_architecture
 - Stability_protocol
 - Encapsulation
 - band_gap_ev_missing
 - perovskite_thickness_nm_missing
 - etl_thickness_nm_missing
 - cell_area_measured_cm2_missing
 - encapsulation_missing
 - light_intensity_missing
 - temperature_missing
 - rh_missing


In [5]:
X = master.drop(columns=forbidden_feature_cols, errors="ignore").copy()
y = master[TARGET_COL].copy()
y_raw = master[RAW_TARGET_COL].copy()
groups = master[GROUP_COL].copy()
row_ids = master[ROW_ID_COL].copy()

print("Initial X shape:", X.shape)
print("y shape:", y.shape)
print("groups shape:", groups.shape)
print("row_ids shape:", row_ids.shape)

Initial X shape: (1835, 50)
y shape: (1835,)
groups shape: (1835,)
row_ids shape: (1835,)


In [6]:
fully_missing_cols = [c for c in X.columns if X[c].isna().all()]

X = X.drop(columns=fully_missing_cols, errors="ignore").copy()

print("Dropped fully missing columns:")
print(fully_missing_cols)
print("X shape after dropping fully missing:", X.shape)

Dropped fully missing columns:
[]
X shape after dropping fully missing: (1835, 50)


In [7]:
constant_cols = [c for c in X.columns if X[c].nunique(dropna=True) <= 1]

X = X.drop(columns=constant_cols, errors="ignore").copy()

print("Dropped constant columns:")
print(constant_cols)
print("X shape after dropping constant columns:", X.shape)

Dropped constant columns:
['protocol_has_o', 'bias_mentions_dark']
X shape after dropping constant columns: (1835, 48)


In [8]:
dup_col_names = pd.Series(X.columns).value_counts()
dup_col_names = dup_col_names[dup_col_names > 1]

print("Duplicate column names:")
print(dup_col_names if len(dup_col_names) > 0 else "None")

Duplicate column names:
None


In [9]:
categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("Categorical columns:", len(categorical_cols))
print("Numeric columns:", len(numeric_cols))
print("\nCategorical sample:", categorical_cols[:20])
print("\nNumeric sample:", numeric_cols[:20])

Categorical columns: 7
Numeric columns: 41

Categorical sample: ['architecture_family', 'etl_family', 'htl_family', 'backcontact_family', 'protocol_family', 'bias_family', 'light_bin']

Numeric sample: ['is_nip', 'is_pin', 'is_other_arch', 'etl_has_tio2', 'etl_has_sno2', 'etl_has_pcbm', 'etl_has_c60', 'etl_has_zno', 'htl_has_spiro', 'htl_has_ptaa', 'htl_has_pedot', 'htl_has_niox', 'htl_has_p3ht', 'back_has_au', 'back_has_ag', 'back_has_al', 'back_has_carbon', 'has_perovskite_additives', 'has_etl_additives', 'has_htl_additives']


C:\Users\khanm\AppData\Local\Temp\ipykernel_2412\3793130707.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()


In [10]:
for col in categorical_cols:
    X[col] = X[col].astype("object").fillna("missing")

print("Filled categorical missing values with 'missing'.")

Filled categorical missing values with 'missing'.


In [11]:
numeric_missing = pd.DataFrame({
    "column": numeric_cols,
    "missing_pct": [100 * X[c].isna().mean() for c in numeric_cols],
    "n_unique": [X[c].nunique(dropna=True) for c in numeric_cols],
    "dtype": [str(X[c].dtype) for c in numeric_cols]
}).sort_values(["missing_pct", "column"], ascending=[False, True])

numeric_missing.to_csv(
    os.path.join(REPORT_DIR, "05_numeric_missingness_before_encoding.csv"),
    index=False
)

numeric_missing.head(30)

,column,missing_pct,n_unique,dtype
22,etl_thickness_nm,84.577657,30,float64
21,perovskite_thickness_nm,65.449591,75,float64
20,band_gap_ev,30.953678,73,float64
36,rh_pct,26.811989,40,float64
35,temperature_c,3.651226,30,float64
31,light_intensity_suns,2.724796,19,float64
23,cell_area_measured_cm2,2.397820,126,float64
14,back_has_ag,0.000000,2,int64
15,back_has_al,0.000000,2,int64
13,back_has_au,0.000000,2,int64


In [12]:
X_encoded = pd.get_dummies(
    X,
    columns=categorical_cols,
    dummy_na=False,
    drop_first=False
)

print("Encoded X shape:", X_encoded.shape)
X_encoded.head(2)

Encoded X shape: (1835, 77)


,is_nip,is_pin,is_other_arch,etl_has_tio2,etl_has_sno2,etl_has_pcbm,etl_has_c60,etl_has_zno,htl_has_spiro,htl_has_ptaa,htl_has_pedot,htl_has_niox,htl_has_p3ht,back_has_au,back_has_ag,back_has_al,back_has_carbon,has_perovskite_additives,has_etl_additives,has_htl_additives,band_gap_ev,perovskite_thickness_nm,etl_thickness_nm,cell_area_measured_cm2,n_cells_per_substrate,encapsulation_flag,protocol_has_l,protocol_has_d,bias_is_mpp,bias_is_oc,bias_is_sc,light_intensity_suns,is_dark_condition,is_approx_1sun,is_high_light,temperature_c,rh_pct,is_room_temperature,is_hot_test,is_dry_condition,is_humid_condition,architecture_family_nip,architecture_family_other_or_unknown,architecture_family_pin,etl_family_c60,etl_family_other_or_unknown,etl_family_other_rare,etl_family_pcbm,etl_family_sno2,etl_family_tio2,etl_family_zno,htl_family_missing,htl_family_niox,htl_family_other_or_unknown,htl_family_other_rare,htl_family_p3ht,htl_family_pedot_pss,htl_family_ptaa,htl_family_spiro_ometad,backcontact_family_ag,backcontact_family_al,backcontact_family_au,backcontact_family_carbon,backcontact_family_cu,backcontact_family_other_rare,protocol_family_isos_d,protocol_family_isos_l,protocol_family_other_isos,protocol_family_other_or_unknown,protocol_family_other_rare,bias_family_mpp,bias_family_open_circuit,bias_family_other_rare,light_bin_dark_or_zero,light_bin_high_light,light_bin_missing,light_bin_other_rare
0,1,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,1.59,NaN,NaN,0.06,0.0,0.0,1,0,1,0,0,100.0,0,0,1,25.0,NaN,1,0,0,0,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False,True,False,False
1,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,NaN,500.0,25.0,0.16,0.0,0.0,1,0,1,0,0,100.0,0,0,1,25.0,NaN,1,0,0,0,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False,True,False,False


In [13]:
# CELL 13 — audit duplicate rows only, DO NOT drop them

dup_mask = X_encoded.duplicated()
n_dup_rows = int(dup_mask.sum())

print("Exact duplicate X rows found:", n_dup_rows)
print("Duplicates are kept in the dataset.")
print("Shapes after duplicate-row audit:")
print("X_encoded:", X_encoded.shape)
print("y:", y.shape)
print("y_raw:", y_raw.shape)
print("groups:", groups.shape)
print("row_ids:", row_ids.shape)

Exact duplicate X rows found: 446
Duplicates are kept in the dataset.
Shapes after duplicate-row audit:
X_encoded: (1835, 77)
y: (1835,)
y_raw: (1835,)
groups: (1835,)
row_ids: (1835,)


In [14]:
# CELL 13A — save duplicate audit table for inspection

dup_audit = X_encoded.copy()
dup_audit["_is_duplicate_x"] = dup_mask.astype(int)
dup_audit["_raw_row_id"] = row_ids.values
dup_audit["_doi"] = groups.values
dup_audit["_T80_log1p"] = y.values
dup_audit["_T80_clean"] = y_raw.values

dup_only = dup_audit[dup_audit["_is_duplicate_x"] == 1].copy()

dup_only.to_csv(
    os.path.join(REPORT_DIR, "05_duplicate_x_rows_audit.csv"),
    index=False
)

print("Duplicate X audit rows saved:", len(dup_only))

Duplicate X audit rows saved: 446


In [15]:
assert len(X_encoded) == len(y) == len(y_raw) == len(groups) == len(row_ids), "Row mismatch after processing"
assert y.isna().sum() == 0, "Target contains missing values"
assert y_raw.isna().sum() == 0, "Raw target contains missing values"
assert groups.isna().sum() == 0, "Groups contain missing values"

print("Final integrity checks passed.")

Final integrity checks passed.


In [16]:
feature_summary = pd.DataFrame({
    "feature": X_encoded.columns,
    "dtype": X_encoded.dtypes.astype(str).values,
    "missing_pct": X_encoded.isna().mean().values * 100,
    "n_unique": [X_encoded[c].nunique(dropna=True) for c in X_encoded.columns]
}).sort_values(["missing_pct", "feature"], ascending=[False, True])

feature_summary.to_csv(
    os.path.join(REPORT_DIR, "05_feature_summary_encoded.csv"),
    index=False
)

feature_summary.head(40)

,feature,dtype,missing_pct,n_unique
22,etl_thickness_nm,float64,84.577657,30
21,perovskite_thickness_nm,float64,65.449591,75
20,band_gap_ev,float64,30.953678,73
36,rh_pct,float64,26.811989,40
35,temperature_c,float64,3.651226,30
31,light_intensity_suns,float64,2.724796,19
23,cell_area_measured_cm2,float64,2.397820,126
41,architecture_family_nip,bool,0.000000,2
42,architecture_family_other_or_unknown,bool,0.000000,2
43,architecture_family_pin,bool,0.000000,2


In [17]:
X_path = os.path.join(INTERIM_DIR, "X_features.parquet")
y_path = os.path.join(INTERIM_DIR, "y_target.parquet")
y_raw_path = os.path.join(INTERIM_DIR, "y_target_raw.parquet")
groups_path = os.path.join(INTERIM_DIR, "groups.parquet")
row_ids_path = os.path.join(INTERIM_DIR, "row_ids.parquet")

X_encoded.to_parquet(X_path, index=False)
pd.DataFrame({"T80_log1p": y}).to_parquet(y_path, index=False)
pd.DataFrame({"T80_clean": y_raw}).to_parquet(y_raw_path, index=False)
pd.DataFrame({"Ref_DOI_number": groups}).to_parquet(groups_path, index=False)
pd.DataFrame({"raw_row_id": row_ids}).to_parquet(row_ids_path, index=False)

print("Saved:", X_path)
print("Saved:", y_path)
print("Saved:", y_raw_path)
print("Saved:", groups_path)
print("Saved:", row_ids_path)

Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\X_features.parquet
Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\y_target.parquet
Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\y_target_raw.parquet
Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\groups.parquet
Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\interim\row_ids.parquet


In [18]:
feature_card = {
    "notebook": "05_prepare_feature_matrix_pooled.ipynb",
    "source_master_file": MASTER_FILE,
    "target_column": TARGET_COL,
    "raw_target_column": RAW_TARGET_COL,
    "group_column": GROUP_COL,
    "row_id_column": ROW_ID_COL,
    "rows_final": int(len(X_encoded)),
    "n_features_final": int(X_encoded.shape[1]),
    "n_groups_final": int(groups.nunique()),
    "dropped_columns": {
        "forbidden_feature_cols": forbidden_feature_cols,
        "fully_missing_cols": fully_missing_cols,
        "constant_cols": constant_cols,
    },
    "categorical_cols_before_encoding": categorical_cols,
    "numeric_cols_before_encoding": numeric_cols,
    "notes": [
    "Target leakage columns removed.",
    "Grouping, row ID, and metadata columns removed from predictors.",
    "Fully missing and constant columns removed.",
    "Categoricals one-hot encoded with missing filled as 'missing'.",
    "Numeric missing values are retained for model-side handling.",
    "Exact duplicate X rows were audited but not removed."
],
    "saved_files": {
        "X_features": X_path,
        "y_target": y_path,
        "y_target_raw": y_raw_path,
        "groups": groups_path,
        "row_ids": row_ids_path,
    }
}

feature_card_path = os.path.join(META_DIR, "feature_card.json")

with open(feature_card_path, "w", encoding="utf-8") as f:
    json.dump(feature_card, f, indent=4)

print("Saved:", feature_card_path)

Saved: C:\Users\khanm\OneDrive\Desktop\3rd model 18-03-2026\metadata\feature_card.json


In [19]:
print("========== FEATURE MATRIX SUMMARY ==========")
print("Master input rows:", len(master))
print("Final X rows:", len(X_encoded))
print("Final X columns:", X_encoded.shape[1])
print("Final y rows:", len(y))
print("Final DOI groups:", groups.nunique())
print("Exact duplicate X rows found:", n_dup_rows)
print("Exact duplicate X rows removed: 0")

print("\nDropped fully missing columns:")
print(fully_missing_cols)

print("\nDropped constant columns:")
print(constant_cols)

print("\nTop 20 remaining feature columns:")
print(X_encoded.columns[:20].tolist())

print("\nSaved final pooled modeling artifacts successfully.")

========== FEATURE MATRIX SUMMARY ==========
Master input rows: 1835
Final X rows: 1835
Final X columns: 77
Final y rows: 1835
Final DOI groups: 964
Exact duplicate X rows found: 446
Exact duplicate X rows removed: 0

Dropped fully missing columns:
[]

Dropped constant columns:
['protocol_has_o', 'bias_mentions_dark']

Top 20 remaining feature columns:
['is_nip', 'is_pin', 'is_other_arch', 'etl_has_tio2', 'etl_has_sno2', 'etl_has_pcbm', 'etl_has_c60', 'etl_has_zno', 'htl_has_spiro', 'htl_has_ptaa', 'htl_has_pedot', 'htl_has_niox', 'htl_has_p3ht', 'back_has_au', 'back_has_ag', 'back_has_al', 'back_has_carbon', 'has_perovskite_additives', 'has_etl_additives', 'has_htl_additives']

Saved final pooled modeling artifacts successfully.
